In [ ]:
print("AH")

In [ ]:
import torch
# This ensures the code works on your laptop even without a GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
%pip install einx

In [ ]:
import torch
import torch.nn as nn
from torch.optim import SGD
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
import os
import torchvision.transforms as transforms
import json
import einx

from tqdm import tqdm

In [ ]:
class COCOObjectDetectionDataset(Dataset):
    def __init__(self, image_dir, annotation_file, transform=None):
        self.image_dir = image_dir
        self.transform = transform or transforms.ToTensor()

        # Load annotations
        with open(annotation_file, 'r') as f:
            data = json.load(f)

        # Hardcode COCO fruit category IDs
        fruit_category_ids = {52, 53, 55}  # banana, apple, orange

        # Map image_id → file_name
        self.imgs = {img['id']: img['file_name'] for img in data['images']}

        # Build mapping: image_id → (bbox, label) for fruit categories only
        self.image_instances = {}
        for ann in tqdm(data['annotations']):
            if ann['category_id'] not in fruit_category_ids:
                continue  # skip non-fruit annotations
            img_id = ann['image_id']
            bbox = ann['bbox']
            label = ann['category_id']
            if img_id not in self.image_instances:
                self.image_instances[img_id] = []
            self.image_instances[img_id].append((bbox, label))

        # Keep only images that contain fruit annotations
        self.image_ids = list(self.image_instances.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.image_dir, self.imgs[img_id])
        image = Image.open(img_path).convert('RGB')
        target = self.image_instances[img_id]

        boxes = [x[0] for x in target]
        labels = [x[1] for x in target]

        if self.transform:
            image = self.transform(image)
        return image, np.array(boxes), labels
        
def get_category_names(annotation_file):
    with open(annotation_file, 'r') as f:
        data = json.load(f)
    return {cat['id']: cat['name'] for cat in data['categories']}
    


train_images_dir = '/kaggle/input/coco-2017-dataset/coco2017/train2017'
train_annotation_file = '/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_train2017.json'
category_names = get_category_names(train_annotation_file)
train_ds = COCOObjectDetectionDataset(train_images_dir, train_annotation_file)


In [ ]:
img, boxes, labels = train_ds[10]
print(boxes)

In [ ]:
def plotImg(img, boxes, labels):

    img_np = einx.rearrange("c h w -> h w c", img).numpy()
    fig, ax = plt.subplots(1)
    ax.imshow(img_np)
    
    for box, label_id in zip(boxes, labels):
        x, y, w, h = box
        rect = plt.Rectangle((x, y), w, h, fill=False, color='red', linewidth=2)
        ax.add_patch(rect)
        # Draw category name on the box
        label_name = category_names[label_id]
        ax.text(x, y, label_name, fontsize=10, color='yellow',
                bbox=dict(facecolor='red', alpha=0.5))
    
    plt.axis('off')
    plt.show()
plotImg(img, boxes, labels)


In [ ]:
from functools import partial

# Depending on how we architect our model, we will want to transform our 
# ground truth data into a grid of (HeightCells, WidthCells) and for each cell
# find what bounding boxes that should be associated with that cell


# To make the math with bounding boxes and intersections over unions much easier, 
# our model will predict these normalized values. But it will become much easier to calculate
# the IOU if we work in pixel space for both the ground truth and our predictions
# So, along with a normalized bounding box, here's a function which can take as input
# that normalized bounding box and conver it to pixel space.
# HINT!: This function can also be used to turn an arbitrary model prediction back into pixel space

def unnormalizeBBox(normalizedBBox, numVerticalCells, numHorizontalCells, cellYIdx, cellXIdx, imgHeight, imgWidth):
    # if normalizedBBox is a tensor, detach it and make it a numpy array
    if hasattr(normalizedBBox, "detach"):
        normalizedBBox = normalizedBBox.detach().cpu().numpy()
    cellWidth = imgWidth / numHorizontalCells
    cellHeight = imgHeight / numVerticalCells
    centerXPixelspace = normalizedBBox[0] * cellWidth + cellXIdx * cellWidth
    centerYPixelspace = normalizedBBox[1] * cellHeight + cellYIdx * cellHeight
    pixelWidth = normalizedBBox[2] * imgHeight
    pixelHeight = normalizedBBox[3] * imgWidth
    return np.array([centerXPixelspace, centerYPixelspace, pixelWidth, pixelHeight])
    
def prepareGroundTruth(img, boxes, labels, numVertCells, numHorizontalCells):
    cellH = (img.shape[1] / numVertCells)
    cellW = (img.shape[2] / numHorizontalCells)
    bboxArray = [[[] for _ in range(numHorizontalCells)] for _ in range(numVertCells)]
    for box in boxes:
        boxCenterWidth = box[0] + box[2] / 2
        boxCenterHeight = box[1] + box[3] / 2

        # Get which cell the bounding box should be placed in 
        cellHeightIdx = int(boxCenterHeight // cellH)
        cellWidthIdx = int(boxCenterWidth // cellW)
        
        # Clip indices to prevent out of range errors
        if cellWidthIdx >= numHorizontalCells:
            cellWidthIdx = numHorizontalCells - 1
        if cellHeightIdx >= numVertCells:
            cellHeightIdx = numVertCells - 1

        # normalize box width/height by img dim
        # normalize center to be offset relative to cell top left corner 

        normalizedCenterX = (boxCenterWidth - cellWidthIdx * cellW) / cellW
        normalizedCenterY = (boxCenterHeight - cellHeightIdx * cellH) / cellH
        normalizedWidth = box[2] / img.shape[2]
        normalizedHeight = box[3] / img.shape[1]

        normalizedBBox = torch.tensor([normalizedCenterX, normalizedCenterY, normalizedWidth, normalizedHeight], dtype=torch.float32)
        
        bboxArray[cellHeightIdx][cellWidthIdx].append(normalizedBBox)
        
    return bboxArray

numVerticalCells, numHorizontalCells = 3, 3

# Passing 3, 3 in as arguments splits the image up into cells like a tic-tac-toe board
arr = prepareGroundTruth(img, boxes, labels, numVerticalCells, numHorizontalCells)

# arr[1][1] gets a list of all the bounding boxes whose centers are in the middle cell
arr[1][1]

In [ ]:
cellYIdx, cellXIdx = 1, 1

firstBoundingBoxMiddleCell = arr[cellYIdx][cellXIdx][0]
print(firstBoundingBoxMiddleCell)

# As you can see, applying our function brings us back nicely into pixel space
# Here the list is, in pixels, [centerX, centerY, width, height]

print(unnormalizeBBox(firstBoundingBoxMiddleCell, 
                      numVerticalCells, 
                      numHorizontalCells, 
                      cellYIdx, 
                      cellXIdx, 
                      img.shape[1], 
                      img.shape[2]))

In [ ]:
# Given two arbitrary boxes in pixel space where each box
# is represented by [centerX, centerY, width, height],
# calculate the intersection over union of the two boxes
# should always return a float between 0 and 1
def IOU(pixelSpaceBox1, pixelSpaceBox2):
    # pixelSpaceBox coordinates array = [x, y, w, h]
    
    # convert center coords to corner corners (top left and bottom right)
    # box 1
    left_x_1 = pixelSpaceBox1[0] - (pixelSpaceBox1[2] / 2)
    top_y_1 = pixelSpaceBox1[1] - (pixelSpaceBox1[3] / 2)
    right_x_1 = pixelSpaceBox1[0] + (pixelSpaceBox1[2] / 2)
    bot_y_1 = pixelSpaceBox1[1] + (pixelSpaceBox1[3] / 2)
    

    # box 2
    left_x_2 = pixelSpaceBox2[0] - (pixelSpaceBox2[2] / 2)
    top_y_2 = pixelSpaceBox2[1] - (pixelSpaceBox2[3] / 2)
    right_x_2 = pixelSpaceBox2[0] + (pixelSpaceBox2[2] / 2)
    bot_y_2 = pixelSpaceBox2[1] + (pixelSpaceBox2[3] / 2)


    # find intersection boundaries
    left_edge_x = max(left_x_1, left_x_2)
    right_edge_x = min(right_x_1, right_x_2)
    top_edge_y = max(top_y_1, top_y_2)
    bot_edge_y = min(bot_y_1, bot_y_2)
    intersectWidth = max(0, right_edge_x - left_edge_x)
    intersectHeight = max(0, bot_edge_y - top_edge_y)
    
    # intersection area
    intersection_area = intersectWidth * intersectHeight

    # area of union (sum of both box areas minus intersection)
    area_box_1 = pixelSpaceBox1[2] * pixelSpaceBox1[3]
    area_box_2 = pixelSpaceBox2[2] * pixelSpaceBox2[3]
    area_of_union = (area_box_1 + area_box_2) - intersection_area

    # IOU = area of intersection / area of union
    IOU = intersection_area / area_of_union
    return float(IOU)


In [ ]:
import torch
# using this to replace calcFullLoss and YOLOLossSingleCell
def vectorized_yolo_loss(predictions, targets, lambda_coord=5.0, lambda_noobj=0.5):
    """
    predictions: (Batch, S, S, B*5 + C) from model
    targets: (Batch, S, S, B, 5) from collate_fn (x, y, w, h, conf)
    """
    # reshape predictions to match target structure [cite: 857, 867]
    # predictions are (Batch, S, S, 10 + classes) then split first 10 values into (Batch, S, S, 2, 5)
    batch_size, S, _, _ = predictions.shape
    pred_boxes = predictions[..., :10].view(batch_size, S, S, 2, 5)

    # LOCALIZATION LOSS (XY)
    # identify where objects exist in ground truth
    # then calculate loss where object exists
    exists_mask = targets[..., 4] == 1.0  # (Batch, S, S, B)
    xy_loss = torch.sum(
        exists_mask.unsqueeze(-1) * (pred_boxes[..., :2] - targets[..., :2])**2
    )
    
    # DIMENSION LOSS (WH)
    wh_loss = torch.sum(
        exists_mask.unsqueeze(-1) * (torch.sqrt(torch.abs(pred_boxes[..., 2:4]) + 1e-6) - 
         torch.sqrt(targets[..., 2:4]))**2
    )
    
    # CONFIDENCE LOSS w object PRESENT
    obj_conf_loss = torch.sum(
        exists_mask * (pred_boxes[..., 4] - targets[..., 4])**2
    )
    
    # CONFIDENCE LOSS w NO object
    noobj_mask = ~exists_mask
    noobj_conf_loss = torch.sum(
        noobj_mask * (pred_boxes[..., 4] - 0.0)**2
    )
    
    total_loss = (lambda_coord * (xy_loss + wh_loss) + 
                  obj_conf_loss + 
                  (lambda_noobj * noobj_conf_loss))
    
    return total_loss / batch_size

In [ ]:
# Define a pytorch dataset where the __getitem__(self, idx) function returns both an image (normalized to, for example, 224 by 224 pixels)
# So shape: (3, 224, 224)
# and a ground truth data that looks kind of like exampleGroundTruthBboxes above. Note that the function
# prepareGroundTruth(img, boxes, labels, numVerticalCells, numHorizontalCells) that I provided in the codestem
# process the coco data into this format very easily given the img, boxes, labels variables returned by my stem dataset

class Pytorch_Dataset(Dataset):
    # set up path
    def __init__(self, annotations_file, img_dir, transform=None):
        self.img_dir = img_dir # image directory
        self.transform = transform
        self.verticalCells = 7
        self.horizontalCells = 7

        # load annotations
        print("Opening JSON file...") # should be a couple mins wait time hopefully
        with open(annotations_file, 'r') as f:
            data = json.load(f)
            
        self.imgs = {img['id']: img['file_name'] for img in data['images']}
        fruit_category_ids = {52, 53, 55} # banana, apple, orange
        self.image_instances = {}
        
        print("Filtering first 500 fruit images...")
        # stop after 500 imgs to save GPU quota
        target_image_count = 500 
        
        for ann in data['annotations']: 
            if ann['category_id'] in fruit_category_ids:
                img_id = ann['image_id']
                if img_id not in self.image_instances:
                    if len(self.image_instances) >= target_image_count:
                        break # stop loop early!
                    self.image_instances[img_id] = []
                self.image_instances[img_id].append((ann['bbox'], ann['category_id']))
                
        self.image_ids = list(self.image_instances.keys())
        print(f"Success! Loaded {len(self.image_ids)} images. Ready to train.")
    
    # total # of samples in dataset
    def __len__(self):
        return len(self.image_ids)
    
    # using given index idx, retrieve specific data item (image + ground truth) corresponding to idx
    # load image, resize, convert to tensor/normalize
    # retrieve + process + return
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.img_dir, self.imgs[img_id])

        raw_image = Image.open(img_path).convert('RGB')
        if self.transform:
            processed_image = self.transform(raw_image)
        else:
            processed_image = transforms.ToTensor()(raw_image)
            
        # get ground truth
        target = self.image_instances[img_id]
        boxes = [x[0] for x in target]
        labels = [x[1] for x in target]
    
        groundTruthBBoxes = prepareGroundTruth(
            img=processed_image, # tensor shape
            boxes=boxes,
            labels=labels,
            numVertCells=self.verticalCells,
            numHorizontalCells=self.horizontalCells
        )
        return processed_image, groundTruthBBoxes


# things alan wants to warn me about
# 1 (bonus points) -- i think i did it?? check groundTruthBBoxes_tensors above
# training runs faster on gpu
    # tensors t1 and t2 --> to move to gpu (if u have access) t1 = t1.kuda or smth (just google how)
    # if u have one tensor on GPU + and one on CPU YOU GET AN ERROR
    # should return torch tensor
    # must be on same device as what tensor ure trying to compute to
# model output is a single tensor (array of numbers, nested lists)
# ground truth is a list of lists --> numpy array inside somewhere
# dont want to hardcode in what device ure using
# device = kuda torch if available
    # move model output to device ure working w --> modeloutput = modeloutput.to(device)
# some func that takes model output + some comparison
    # might show up in normalizing needing to turn into tensor --> tensor ground truth = torch tensor of array
    # must be moved to same device as model output
        # device parameter (everything goes to device)
# after u move to same device THEN you can compare (operations) w/o errors
# device = place where computations occur (gpu or cpu -- gpu superior for this)
# dont use numpy it doesn't nicely interface w gpu --> use tensor for torch

In [ ]:
# 2 (try)
# how to add collate func + understand it -- add after pytorch
# collate make ur own func
    # for these lists of lists -- given two lists return A list
        # joining 2 sampels in a list = collating things together
        # pytorch likes to make a stack of things like tensors but we don't want that so u should
            # define ur own func
# dataloaders (check dataloaders pytorch documentation) --> they have a whole section on working
    # w a collate function
    # ask llm for simple examples for custom collate function + working w dataloader
    # youll know if it works if u can sample a batch of data without getting an error

"""takes lists and make single batch tensor"""
def yolo_collate_fn(batch, num_cells=7, num_boxes=2):
    """
    batch: list of tuples (image_tensor, ground_truth_list_of_tensors)
    """
    images = []
    # [Batch, S, S, B, 5] -> (x, y, w, h, conf)
    # B = 2 to match model's output design
    target_tensor = torch.zeros((len(batch), num_cells, num_cells, num_boxes, 5))

    for batch_idx, (img, gt_boxes_list) in enumerate(batch):
        images.append(img)
        
        # gt_boxes_list is S x S list of lists from prepareGroundTruth
        for row_idx, row in enumerate(gt_boxes_list):
            for col_idx, cell_boxes in enumerate(row):
                for box_idx, box_coords in enumerate(cell_boxes):
                    if box_idx < num_boxes:
                        # box_coords [x, y, w, h]
                        # fill first 4 vals + set confidence (index 4) to 1.0
                        target_tensor[batch_idx, row_idx, col_idx, box_idx, :4] = box_coords
                        target_tensor[batch_idx, row_idx, col_idx, box_idx, 4] = 1.0

    return torch.stack(images), target_tensor

In [ ]:
import torchvision.transforms as transforms

# defines var for test_transform
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# initialize REAL dataset for training
train_images_dir = '/kaggle/input/coco-2017-dataset/coco2017/train2017'
train_annotation_file = '/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_train2017.json'

# create dataset object
test_ds = Pytorch_Dataset(
    annotations_file=train_annotation_file, 
    img_dir=train_images_dir,
    transform=test_transform
)

In [ ]:
# loop through data to train

from torch.utils.data import DataLoader

train_loader = DataLoader(
    test_ds, 
    batch_size=16, 
    shuffle=True, 
    collate_fn=lambda b: yolo_collate_fn(b, num_cells=test_ds.verticalCells)
)

# Testing the batching
images, targets = next(iter(train_loader))
print(f"Images batch shape: {images.shape}")  # Expected: [16, 3, 224, 224]
print(f"Targets batch shape: {targets.shape}") # Expected: [16, 7, 7, 2, 5]

In [ ]:
import einx
# Define a convolutional neural network which takes in an image of of shape
# (3, 224, 224) and outputs a tensor of shape (numVerticalGridCells, numHorizontalGridCells, numBoundingBoxesPerGridCell, 5)
# A Conv2d Layer takes you from an image of shape (channelsIn, height, width) -> (channelsOut, heightOut, widthOut)
# Instead use the neural network to go from (as in the paper):
# (3, 224, 224) -> (numChannelsOut, heightOut, widthOut)
# (3, 224, 224) -> (5 * numBoundingBoxesPerCell, numVerticalGridCells, numHorizontalGridCells)

# 24 convolutional layers + 2 fully connected layers

# alternate 1x1 and 3x3 blocks
def YOLO_helper_repeat_blocks(channels, layers_config):
    # make sequence of repeating blocks
    # layers config is list of tuples
        # 1x1, 3x3, how many times to repeat

    layers = []
    curr_channels = channels

    for out_channel_1, out_channel_3, repeat in layers_config:
        for i in range(repeat):
            # 1x1
            # define new input space for following 3x3 conv
            layers.append(
                nn.Conv2d(
                    curr_channels,
                    out_channel_1,
                    kernel_size = 1
                )
            )
            layers.append(nn.LeakyReLU(0.1))

            # 3x3
            layers.append(
                nn.Conv2d(
                    out_channel_1,
                    out_channel_3,
                    kernel_size = 3,
                    padding = 1
                )
            )

            curr_channels = out_channel_3

        return nn.Sequential(*layers), curr_channels



# NOTE: couldn't figure out how to use nn.module so i just stuck w nn.sequential for now???
            # list of layers, append every block then wrap it in nn.Module list
                # ask llm for usage / implementation
                # use for forward function to iterate through list
            # self.features is an nn.module of the list kinda

class YOLO_CNN(nn.Module):

    def __init__(self, S, B):
        super().__init__()
        self.flatten = nn.Flatten()
        self.S = 7
        self.B = 2
        self.C = 20
        C_out = B * 5
        C_out_total = C_out + self.C

        # initial 4 layers
            # conv 7x7x64-s-2
            # maxpool 2x2-s-2
            # conv 3x3x192
            # maxpool 2x2-s-2
        initial_layers = nn.Sequential(
            # 3 RGB channels
            nn.Conv2d(3, 64, kernel_size = 7, stride = 2, padding = 3),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size = 2, stride = 2),

            nn.Conv2d(64, 192, kernel_size = 3, padding = 1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size = 2, stride = 2),
        )

        # repeated layers
            # pattern is always 1x1 then 3x3
            # third number changes = output channel
            # output channel 1x1, output channel 3x3, repeat
        # 128, 256, 1
        # 256, 512, 1
        # maxpool layer
        config_repeat_1 = [
            (128, 256, 1),
            (256, 512, 1),
        ]
        repeat_layers_1, final_channels_1 = YOLO_helper_repeat_blocks(
            192, config_repeat_1
        )
            # 192 initial channels
        maxpool_3 = nn.MaxPool2d(kernel_size = 2, stride = 2)

        # 256, 512, 4
        # 512, 1024, 1
        # maxpool layer
        config_repeat_2 = [
            (256, 512, 4),
            (512, 1024, 1),
        ]
        repeat_layers_2, final_channels_2 = YOLO_helper_repeat_blocks(
            final_channels_1, config_repeat_2
        )

        # 512, 1024, 2
        config_repeat_3 = [
            (512, 1024, 2),
        ]
        repeat_layers_3, final_channels_3 = YOLO_helper_repeat_blocks(
            final_channels_2, config_repeat_3
        )

        final_channels = 1024
        # final layers -- last 4??
            # 3x3x1024
            # 3x3x1024-s-2 (maxpool kinda)
            # 3x3x1024
            # 3x3x1024
        self.final_layers = nn.Sequential(
            nn.Conv2d(final_channels, 1024, kernel_size = 3, padding = 1),
            nn.LeakyReLU(0.1),
            
            nn.Conv2d(1024, 1024, kernel_size = 3, stride = 2, padding = 1),
            nn.LeakyReLU(0.1),

            nn.Conv2d(1024, 1024, kernel_size = 3, padding = 1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 1024, kernel_size = 3, padding = 1),
            nn.LeakyReLU(0.1),
        )

        # leaky rectified linear activation -- linear activation function for final layer
        # conv layer 7x7x64-s-2 (s-2 indicating stride of 2)
        # 3x3x1024 = kernel size x channel depth (final feature map)
        self.features = nn.Sequential(
            initial_layers,
            repeat_layers_1,
            maxpool_3,
            repeat_layers_2,
            repeat_layers_3,
            self.final_layers
        )

        # googLe net 2 fully connected layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.S * self.S * 1024, 4096),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.5),

            # final layer + uses its own linear activation
            nn.Linear(4096, self.S * self.S * (self.B * 5 + self.C))
        )


    # NN raw output = # channels output * output height * output width
    # height + width correspond to vertical + horiz cell counts
    # channels out = number of bounding boxes per cell x 5
    # s x s x b x d = SVertical * SHorizontal * bbox * 5
    
    # After the neural network output, reshape to the proper size:
    # einx.rearrange("(B d) S_V S_H -> S_V S_H B d", modelOutput)

    def forward(self, x):
        # extract features + raw prediction
        x = self.features(x)
        raw_pred = self.classifier(x)

        C_out_total = (self.B * 5) + self.C
        x = raw_pred.view(-1, self.S, self.S, (self.B * 5) + self.C)
        return x

        # reshape the output to fit required format for iou + yolo loss func
        # take raw output x, plug in formula, total @ of prediction channels P
        final_prediction = einx.rearrange(
            x, 
            'b P S_V S_H -> b S_V S_H P', 
            P = C_out_total,
            S_V = self.S,
            S_H = self.S
        )
        
        # Output shape: (BatchSize, S, S, B*5 + C) 
        # E.g., (BatchSize, 7, 7, 30)
        return final_prediction


exampleModelOutput = torch.rand(5 * 2, 3, 3)
print(exampleModelOutput.shape)

transformedModelOutput = einx.rearrange("(B d) S_V S_H -> S_V S_H B d", exampleModelOutput,
                                        B = 2, d=5, S_V=3, S_H = 3)
print(transformedModelOutput.shape)

In [ ]:
import torch.optim as optim

# training loop
# set up necessities, input data, train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = YOLO_CNN(S=7, B=2).to(device) 
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training Loop
print("Starting Training...")
model.train()

for epoch in range(50):
    epoch_loss = 0
    # dummy loader
    for batch_idx, (images, targets) in enumerate(train_loader):
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        predictions = model(images)
        
        loss = vectorized_yolo_loss(predictions, targets)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx} | Loss: {loss.item():.4f}")

    print(f"Epoch {epoch+1} Complete. Avg Loss: {epoch_loss/len(train_loader):.4f}")

# save model
torch.save(model.state_dict(), 'yolo_fruit_model.pth')
print("Model saved as yolo_fruit_model.pth")

In [ ]:
print(f"Total fruit images found: {len(test_ds)}") # coule thousands
img, target = test_ds[0]
print(f"Total fruit images found: {len(test_ds)}")
print(f"Image shape: {img.shape}")

# target is a list
# check length / convert to see dimensions
print(f"Target grid size: {len(target)}x{len(target[0])}")

In [ ]:
"""
def get_all_predictions(model_output, threshold=0.4):
    ""
    takes model output (1, 7, 7, 30) + returns most confident box
        model_output (Batch, S, S, B*5 + C)
        focus on first 10 values (2 boxes * 5 parameters)
        find box with highest confidence score (index 4)
    
    returns a list of ALL boxes that exceed confidence threshold
    ""
    # reshape to (7, 7, 2, 5) --> access each of 98 potential boxes
    pred_boxes = model_output[0, :, :, :10].view(7, 7, 2, 5)
    
    found_boxes = []
    
    for y in range(7):
        for x in range(7):
            # check both boxes in each cell
            for b in range(2):
                conf = pred_boxes[y, x, b, 4].item()
                if conf >= threshold:
                    pixel_box = unnormalizeBBox(
                        pred_boxes[y, x, b, :4], 7, 7, y, x, 224, 224
                    )
                    found_boxes.append((pixel_box, conf))
                    
    return found_boxes
"""

device = torch.device("cpu") # For laptop use
# Make sure this name matches your class definition exactly!
model = YOLO_CNN(S=7, B=2).to(device) 
model.load_state_dict(torch.load('yolo_fruit_model.pth', map_location=device))
model.eval()

def get_best_prediction(output):
    output = output.squeeze(0).cpu() 
    # Index 4 is the universal "Objectness" score in your architecture
    # We find the one cell in the 7x7 grid that is MOST sure there is an object
    obj_map = torch.sigmoid(output[:, :, 4])
    max_idx = torch.argmax(obj_map)
    i, j = max_idx // 7, max_idx % 7
    conf = obj_map[i, j].item()

    # Get coordinates and scale relative to grid cell (Fix for tracking)
    x_off = torch.sigmoid(output[i, j, 0]).item()
    y_off = torch.sigmoid(output[i, j, 1]).item()
    w_val = torch.sigmoid(output[i, j, 2]).item()
    h_val = torch.sigmoid(output[i, j, 3]).item()

    # The math that makes the box move correctly across the screen
    x_img = (j + x_off) / 7.0
    y_img = (i + y_off) / 7.0
    return ([x_img, y_img, w_val, h_val], conf)

In [ ]:
# inference + visualization
def predict_and_plot(model, dataset, idx=0):
    model.eval()
    img, _ = dataset[idx] # get a real img
    
    input_tensor = img.unsqueeze(0).to(device)
    with torch.no_grad():
        prediction = model(input_tensor)
    
    # get best bbox from 7x7x2 grid then unnormalize to pixel
    pixel_box, confidence = get_best_prediction(prediction)
    
    if pixel_box is not None:
        print(f"Fruit detected with {confidence*100:.1f}% confidence!")
        # [cite_start]Re-use your plotImg function to show the result [cite: 111-125]
        plotImg(img, [pixel_box], ["Predicted Fruit"])
    else:
        print("No fruit detected above threshold.")

# Run this after training
# predict_and_plot(model, test_ds, idx=5)

In [ ]:
# live video camera feed
import torch
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms

# load the model, then resize the inputed images
# presdict bbox
# find best bbox then draw

# UPDATE START OF CAMERA CELL
device = torch.device("cpu") # CPU for laptop
model = YOLO_CNN(S=7, B=2).to(device)

# Load weights
model.load_state_dict(torch.load('yolo_fruit_model.pth', map_location=device))
model.eval()

print("Model loaded successfully! Starting camera...")
cap = cv2.VideoCapture(0)

fruit_classes = ["Apple", "Banana", "Orange", "Fruit"]

cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret: break
    
    # Pre-process: Matches your training exactly
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img_t = transforms.ToTensor()(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        out = model(img_t)

    box, conf = get_best_prediction(out)

    # Only show the box if the model is at least a little sure
    if conf > 0.05:
        h_f, w_f, _ = frame.shape
        x_n, y_n, w_n, h_n = box
        
        # Scale back to pixel dimensions
        cx, cy = int(x_n * w_f), int(y_n * h_f)
        bw, bh = int(w_n * w_f), int(h_n * h_f)
        
        cv2.rectangle(frame, (cx-bw//2, cy-bh//2), (cx+bw//2, cy+bh//2), (0, 255, 0), 2)
        cv2.putText(frame, f"Detected: {conf:.2f}", (cx-bw//2, cy-bh//2-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow('Fruit Detector', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# test run -- chatgpt mock run

# Create fake data to test the pipeline logic
batch_size = 4
fake_images = torch.rand(batch_size, 3, 224, 224).to(device)
# Targets: [Batch, S, S, B, 5] -> (x, y, w, h, confidence)
fake_targets = torch.zeros(batch_size, 7, 7, 2, 5).to(device)
fake_targets[:, 3, 3, 0, 4] = 1.0  # Set one fake object in the middle cell

# Single step test
model.train()
optimizer.zero_grad()
preds = model(fake_images)
loss = vectorized_yolo_loss(preds, fake_targets)
loss.backward()
optimizer.step()

print(f"Test Successful! Loss: {loss.item():.4f}")

In [ ]:
# continued mock

# Create a tiny 1-image dataset manually to test the model
fake_img = torch.rand(3, 224, 224)
# Create a fake 7x7 grid target
fake_target = torch.zeros(7, 7, 2, 5) 
fake_target[3, 3, 0, 4] = 1.0 # Put a fake fruit in the middle

print("Testing Model Forward Pass...")
model.eval()
with torch.no_grad():
    prediction = model(fake_img.unsqueeze(0).to(device))
    print(f"Prediction Shape: {prediction.shape}") # Should be [1, 7, 7, 30]

print("Testing Loss Function...")
loss = vectorized_yolo_loss(prediction, fake_target.unsqueeze(0).to(device))
print(f"Loss calculated: {loss.item():.4f}")

In [ ]:
from IPython.display import FileLink
# clickable link to download trained model weights
FileLink(r'yolo_fruit_model.pth')

In [ ]:
import torch
import cv2
import torchvision.transforms as transforms

# 1. SETUP MODEL
device = torch.device("cpu")

# REMOVED 'C' TO FIX THE TYPEERROR
model = YOLO_CNN(S=7, B=2).to(device) 

try:
    # Loading your specific weight file
    model.load_state_dict(torch.load('yolo_fruit_model.pth', map_location=device))
    model.eval()
    print("✅ SUCCESS: Model loaded and ready for presentation!")
except Exception as e:
    print(f"❌ Load Error: {e}")

# 2. FAIL-PROOF MATH FOR 7x7 GRID
def get_box(output):
    output = output.squeeze(0).cpu() 
    # Index 4 is the objectness/confidence score
    conf_map = torch.sigmoid(output[:, :, 4])
    max_idx = torch.argmax(conf_map)
    i, j = int(max_idx // 7), int(max_idx % 7)
    
    # Coordinates from indices 0-3
    x_off = torch.sigmoid(output[i, j, 0]).item()
    y_off = torch.sigmoid(output[i, j, 1]).item()
    w_val = torch.sigmoid(output[i, j, 2]).item()
    h_val = torch.sigmoid(output[i, j, 3]).item()

    # The specific math that makes the box move across your screen
    x_img = (j + x_off) / 7.0
    y_img = (i + y_off) / 7.0
    
    return ([x_img, y_img, w_val, h_val], conf_map[i, j].item())

# 3. LIVE CAMERA LOOP
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret: break
    
    # Pre-process: Matches your training exactly
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img_t = transforms.ToTensor()(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        out = model(img_t)

    box, conf = get_box(out)

    # Threshold 0.1 so it tracks the 'strongest' signal in the dark room
    if conf > 0.1:
        h_f, w_f, _ = frame.shape
        cx, cy = int(box[0] * w_f), int(box[1] * h_f)
        bw, bh = int(box[2] * w_f), int(box[3] * h_f)
        
        # Draw the box
        cv2.rectangle(frame, (cx-bw//2, cy-bh//2), (cx+bw//2, cy+bh//2), (0, 255, 0), 2)
        cv2.putText(frame, f"Apple: {conf:.2f}", (cx-bw//2, cy-bh//2-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow('Fruit Detector', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()